# Feature engineering

In [48]:
import pandas as pd
import numpy as np

daily_sales = pd.read_csv(
    "../data/processed/daily_sales.csv",
    parse_dates=["Date"]
)

print(daily_sales.head())
print(daily_sales.dtypes)

        Date   Revenue  Quantity  Orders  Customers
0 2009-12-01  54351.23     26098     119         91
1 2009-12-02  63172.58     31804     115         94
2 2009-12-03  73972.45     49221     124        106
3 2009-12-04  40582.32     21210      89         76
4 2009-12-05   9803.05      5119      30         26
Date         datetime64[ns]
Revenue             float64
Quantity              int64
Orders                int64
Customers             int64
dtype: object


In [49]:
full_dates = pd.date_range(
    start=daily_sales["Date"].min(),
    end=daily_sales["Date"].max(),
    freq="D"
)

daily_sales = (
    daily_sales
    .set_index("Date")
    .reindex(full_dates)
    .rename_axis("Date")
    .reset_index()
)

daily_sales["Revenue"] = daily_sales["Revenue"].fillna(0)

print("Dataset shape:", daily_sales.shape)
print(daily_sales.head())

Dataset shape: (739, 5)
        Date   Revenue  Quantity  Orders  Customers
0 2009-12-01  54351.23   26098.0   119.0       91.0
1 2009-12-02  63172.58   31804.0   115.0       94.0
2 2009-12-03  73972.45   49221.0   124.0      106.0
3 2009-12-04  40582.32   21210.0    89.0       76.0
4 2009-12-05   9803.05    5119.0    30.0       26.0


### **Feature 1 - Time features**

In [50]:
daily_sales["Year"] = daily_sales["Date"].dt.year
daily_sales["Month"] = daily_sales["Date"].dt.month
daily_sales["DayOfWeek"] = daily_sales["Date"].dt.dayofweek
daily_sales["DayOfMonth"] = daily_sales["Date"].dt.day

### **Cyclical Time Features**


In [51]:
daily_sales["MonthSin"] = np.sin(
    2 * np.pi * daily_sales["Month"] / 12
)

daily_sales["MonthCos"] = np.cos(
    2 * np.pi * daily_sales["Month"] / 12
)

daily_sales["DayOfWeekSin"] = np.sin(
    2 * np.pi * daily_sales["DayOfWeek"] / 7
)

daily_sales["DayOfWeekCos"] = np.cos(
    2 * np.pi * daily_sales["DayOfWeek"] / 7
)

### **Feature 2 - Weekend indicator**

In [52]:
daily_sales["IsWeekend"] = (
    daily_sales["DayOfWeek"] >= 5
).astype(int)

### **Lag Features**

In [53]:
daily_sales["Lag_1"] = (
    daily_sales["Revenue"].shift(1)
)

### **Previous 7 day revenue**

In [54]:
daily_sales["Lag_7"] = (
    daily_sales["Revenue"].shift(7)
)

### **Previous 14 day revenue**

In [55]:
daily_sales["Lag_14"] = (
    daily_sales["Revenue"].shift(14)
)

### **Previous 28 day revenue**

In [56]:
daily_sales["Lag_28"] = (
    daily_sales["Revenue"].shift(28)
)

### **Rolling features**

In [57]:
daily_sales["RollingMean_7"] = (
    daily_sales["Revenue"]
    .shift(1)
    .rolling(7)
    .mean()
)

daily_sales["RollingMean_14"] = (
    daily_sales["Revenue"]
    .shift(1)
    .rolling(14)
    .mean()
)

daily_sales["RollingMean_28"] = (
    daily_sales["Revenue"]
    .shift(1)
    .rolling(28)
    .mean()
)

In [58]:
print(daily_sales.columns.tolist())

['Date', 'Revenue', 'Quantity', 'Orders', 'Customers', 'Year', 'Month', 'DayOfWeek', 'DayOfMonth', 'MonthSin', 'MonthCos', 'DayOfWeekSin', 'DayOfWeekCos', 'IsWeekend', 'Lag_1', 'Lag_7', 'Lag_14', 'Lag_28', 'RollingMean_7', 'RollingMean_14', 'RollingMean_28']


In [59]:
daily_sales["RollingStd_7"] = (
    daily_sales["Revenue"]
    .shift(1)
    .rolling(7)
    .std()
)

### **Forecast dataset**

In [60]:
forecast_df = daily_sales[
    [
        "Date",
        "Year",
        "Month",
        "DayOfWeek",
        "DayOfMonth",
        "IsWeekend",
        "MonthSin",
        "MonthCos",
        "DayOfWeekSin",
        "DayOfWeekCos",
        "Lag_1",
        "Lag_7",
        "Lag_14",
        "Lag_28",
        "RollingMean_7",
        "RollingMean_14",
        "RollingMean_28",
        "RollingStd_7",
        "Revenue"
    ]
].copy()

forecast_df = (
    forecast_df
    .dropna()
    .reset_index(drop=True)
)

print("Forecasting dataset shape:", forecast_df.shape)

print("\nColumns:")
print(forecast_df.columns.tolist())

print("\nMissing values:")
print(forecast_df.isnull().sum())

forecast_df.head()

Forecasting dataset shape: (711, 19)

Columns:
['Date', 'Year', 'Month', 'DayOfWeek', 'DayOfMonth', 'IsWeekend', 'MonthSin', 'MonthCos', 'DayOfWeekSin', 'DayOfWeekCos', 'Lag_1', 'Lag_7', 'Lag_14', 'Lag_28', 'RollingMean_7', 'RollingMean_14', 'RollingMean_28', 'RollingStd_7', 'Revenue']

Missing values:
Date              0
Year              0
Month             0
DayOfWeek         0
DayOfMonth        0
IsWeekend         0
MonthSin          0
MonthCos          0
DayOfWeekSin      0
DayOfWeekCos      0
Lag_1             0
Lag_7             0
Lag_14            0
Lag_28            0
RollingMean_7     0
RollingMean_14    0
RollingMean_28    0
RollingStd_7      0
Revenue           0
dtype: int64


,Date,Year,Month,DayOfWeek,DayOfMonth,IsWeekend,MonthSin,MonthCos,DayOfWeekSin,DayOfWeekCos,Lag_1,Lag_7,Lag_14,Lag_28,RollingMean_7,RollingMean_14,RollingMean_28,RollingStd_7,Revenue
0,2009-12-29,2009,12,1,29,0,-2.449294e-16,1.000000,0.781831,0.623490,0.0,28017.28,50262.29,54351.23,5552.375714,17320.607143,29374.426786,10699.487850,0.0
1,2009-12-30,2009,12,2,30,0,-2.449294e-16,1.000000,0.974928,-0.222521,0.0,10849.35,52545.55,63172.58,1549.907143,13730.443571,27433.311429,4100.668855,0.0
2,2009-12-31,2009,12,3,31,0,-2.449294e-16,1.000000,0.433884,-0.900969,0.0,0.00,30638.25,73972.45,0.000000,9977.190000,25177.147857,0.000000,0.0
3,2010-01-01,2010,1,4,1,0,5.000000e-01,0.866025,-0.433884,-0.900969,0.0,0.00,42470.29,40582.32,0.000000,7788.743571,22535.274643,0.000000,0.0
4,2010-01-02,2010,1,5,2,1,5.000000e-01,0.866025,-0.974928,-0.222521,0.0,0.00,0.00,9803.05,0.000000,4755.151429,21085.906071,0.000000,0.0


In [61]:
forecast_df = (
    forecast_df
    .dropna()
    .reset_index(drop=True)
)

In [62]:
print("Forecasting dataset shape:", forecast_df.shape)

print("\nColumns:")
print(forecast_df.columns.tolist())

print("\nMissing values:")
print(forecast_df.isnull().sum())

forecast_df.head()

Forecasting dataset shape: (711, 19)

Columns:
['Date', 'Year', 'Month', 'DayOfWeek', 'DayOfMonth', 'IsWeekend', 'MonthSin', 'MonthCos', 'DayOfWeekSin', 'DayOfWeekCos', 'Lag_1', 'Lag_7', 'Lag_14', 'Lag_28', 'RollingMean_7', 'RollingMean_14', 'RollingMean_28', 'RollingStd_7', 'Revenue']

Missing values:
Date              0
Year              0
Month             0
DayOfWeek         0
DayOfMonth        0
IsWeekend         0
MonthSin          0
MonthCos          0
DayOfWeekSin      0
DayOfWeekCos      0
Lag_1             0
Lag_7             0
Lag_14            0
Lag_28            0
RollingMean_7     0
RollingMean_14    0
RollingMean_28    0
RollingStd_7      0
Revenue           0
dtype: int64


,Date,Year,Month,DayOfWeek,DayOfMonth,IsWeekend,MonthSin,MonthCos,DayOfWeekSin,DayOfWeekCos,Lag_1,Lag_7,Lag_14,Lag_28,RollingMean_7,RollingMean_14,RollingMean_28,RollingStd_7,Revenue
0,2009-12-29,2009,12,1,29,0,-2.449294e-16,1.000000,0.781831,0.623490,0.0,28017.28,50262.29,54351.23,5552.375714,17320.607143,29374.426786,10699.487850,0.0
1,2009-12-30,2009,12,2,30,0,-2.449294e-16,1.000000,0.974928,-0.222521,0.0,10849.35,52545.55,63172.58,1549.907143,13730.443571,27433.311429,4100.668855,0.0
2,2009-12-31,2009,12,3,31,0,-2.449294e-16,1.000000,0.433884,-0.900969,0.0,0.00,30638.25,73972.45,0.000000,9977.190000,25177.147857,0.000000,0.0
3,2010-01-01,2010,1,4,1,0,5.000000e-01,0.866025,-0.433884,-0.900969,0.0,0.00,42470.29,40582.32,0.000000,7788.743571,22535.274643,0.000000,0.0
4,2010-01-02,2010,1,5,2,1,5.000000e-01,0.866025,-0.974928,-0.222521,0.0,0.00,0.00,9803.05,0.000000,4755.151429,21085.906071,0.000000,0.0


In [63]:
forecast_df.to_csv(
    "../data/processed/forecast_dataset.csv",
    index=False
)